# 04 — ESMFold Validation

Batch validation of ProteinMPNN-designed binder sequences using ESMFold.

Inputs:
- `data/results/proteinmpnn/all_mpnn_sequences.csv`

Outputs:
- Predicted binder PDBs
- pLDDT summary tables

In [1]:
# ============================================================
# Cell 0: Install ESMFold engine
# ============================================================

import os
import time

MODEL_NAME = "esmfold.model"

if not os.path.isfile(MODEL_NAME):

    print("Installing aria2...")
    os.system("apt-get install aria2 -qq")

    print("Downloading ESMFold model weights...")
    os.system(
        f"aria2c -q -x 16 "
        f"https://colabfold.steineggerlab.workers.dev/esm/{MODEL_NAME} &"
    )

    if not os.path.isfile("finished_install"):

        print("Installing Python dependencies...")
        os.system(
            "pip install -q "
            "omegaconf pytorch_lightning biopython "
            "ml_collections einops py3Dmol modelcif"
        )

        print("Installing NVIDIA dllogger...")
        os.system(
            "pip install -q "
            "git+https://github.com/NVIDIA/dllogger.git"
        )

        print("Installing OpenFold...")
        os.system(
            "pip install -q "
            "git+https://github.com/sokrypton/openfold.git"
        )

        print("Installing ESMFold...")
        os.system(
            "pip install -q "
            "git+https://github.com/sokrypton/esm.git"
        )

        os.system("touch finished_install")

    while not os.path.isfile(MODEL_NAME):
        time.sleep(5)

    while os.path.isfile(f"{MODEL_NAME}.aria2"):
        time.sleep(5)

print("✓ ESMFold dependencies installed")
print("✓ ESMFold model weights available")

Installing aria2...
Installing Python dependencies...
Installing NVIDIA dllogger...
Installing OpenFold...
Installing ESMFold...
✓ ESMFold dependencies installed
✓ ESMFold model weights available


In [2]:
# ============================================================
# Cell 1: Imports
# ============================================================

import os
import gc
import re
import hashlib

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from scipy.special import softmax
from jax.tree_util import tree_map

print("✓ Imports loaded")

✓ Imports loaded


In [3]:
# ============================================================
# Cell 2: Connect to project and reload ProteinMPNN outputs
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/Othercomputers/My Mac/pdl1-mini-binder-design"

SEQUENCE_CSV = f"{PROJECT_DIR}/data/results/proteinmpnn/all_mpnn_sequences.csv"
OUTPUT_DIR = f"{PROJECT_DIR}/data/results/esmfold_predictions"

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_sequences = pd.read_csv(SEQUENCE_CSV)

print(f"Loaded {len(df_sequences)} ProteinMPNN sequences")
print(df_sequences.columns.tolist())

required_cols = [
    "backbone",
    "binder_sequence",
    "binder_length",
    "mpnn_score",
    "global_score"
]

missing = [c for c in required_cols if c not in df_sequences.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

display(df_sequences.head())

Mounted at /content/drive
Loaded 216 ProteinMPNN sequences
['backbone', 'header', 'full_sequence', 'target_sequence', 'binder_sequence', 'full_length', 'binder_length', 'mpnn_score', 'global_score', 'sequence_recovery']


,backbone,header,full_sequence,target_sequence,binder_sequence,full_length,binder_length,mpnn_score,global_score,sequence_recovery
0,design_0_0,">design_0_0, score=2.1421, global_score=1.8425...",GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG...,NaN,GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG...,70,70,2.1421,1.8425,NaN
1,design_0_0,">T=0.1, sample=1, score=0.9648, global_score=1...",MAASDKLAAALKDLQALADAAIASGALAEGAAALRARGEAAARELE...,NaN,MAASDKLAAALKDLQALADAAIASGALAEGAAALRARGEAAARELE...,70,70,0.9648,1.3977,0.0714
2,design_0_0,">T=0.1, sample=2, score=0.9330, global_score=1...",MEASDKLAEALKALKAAADAAIASGDAAAGAAAVAAAGAAAAKELA...,NaN,MEASDKLAEALKALKAAADAAIASGDAAAGAAAVAAAGAAAAKELA...,70,70,0.9330,1.3811,0.0714
3,design_0_0,">T=0.1, sample=3, score=0.9404, global_score=1...",MEVSDRLAAALAELQALADAARASGDLAAGRAALAARGEAAAEELE...,NaN,MEVSDRLAAALAELQALADAARASGDLAAGRAALAARGEAAAEELE...,70,70,0.9404,1.3837,0.0571
4,design_0_0,">T=0.1, sample=4, score=0.8645, global_score=1...",MEASDKLAAALAALAAAAAAAAAAGDLAAGRAALAAAGDAAAAELA...,NaN,MEASDKLAAALAALAAAAAAAAAAGDLAAGRAALAAAGDAAAAELA...,70,70,0.8645,1.3683,0.0571


In [5]:
# ============================================================
# Cell 3: Select candidates for ESMFold validation
# ============================================================

TOP_N = 10

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_candidates = (
    df_sequences
    .sort_values("mpnn_score")
    .drop_duplicates("backbone")
    .head(TOP_N)
    .reset_index(drop=True)
)

candidate_csv = f"{OUTPUT_DIR}/esmfold_input_candidates.csv"
df_candidates.to_csv(candidate_csv, index=False)

print(f"Selected {len(df_candidates)} candidates")
print(f"Saved candidate list to: {candidate_csv}")

display(
    df_candidates[
        [
            "backbone",
            "binder_length",
            "mpnn_score",
            "global_score",
            "binder_sequence"
        ]
    ]
)

Selected 10 candidates
Saved candidate list to: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/esmfold_predictions/esmfold_input_candidates.csv


,backbone,binder_length,mpnn_score,global_score,binder_sequence
0,design_9_0,70,0.8210,1.3588,AAVAAAVAQAHLALARAVADASPEEQAAAAARANAAAVAAAADPAA...
1,design_19_0,70,0.8493,1.3756,SEAARAAAAAAAAAGAAAAAATACPAAAAAAAAAAAGAAAASAEGG...
2,design_0_0,70,0.8645,1.3683,MEASDKLAAALAALAAAAAAAAAAGDLAAGRAALAAAGDAAAAELA...
3,design_7_0,70,0.8931,1.3808,AAAARAALLAAAAAAAAAAAAATAALAAAAAELTAAGNAAGAAALK...
4,design_6_0,70,0.9113,1.3849,ATCAALAERAEAAAKAYGLALVAGNATAAAEAAAAAAAVLAEAAAD...
5,design_4_0,70,0.9173,1.3692,SEEKRRAAEEARARARERIAALDPANAERIMAAIEEVSPGGARAIG...
6,design_18_0,70,0.9340,1.4174,AAAAAAAAAAAAALASAVAGAAAAGELNPKAKPAVDAAIAALAAAF...
7,design_22_0,70,0.9394,1.4228,LDEQDARACLRAAENALAAAKANPATALRDLVAAAWFLARAAEALD...
8,design_23_0,70,0.9519,1.4143,MGEGVRKVGEGCELLRRAVALAPEDPAAAAALAARGDELIAAGMAT...
9,design_1_0,70,0.9583,1.4177,SEAYKQALATLAAGIAAATKAAATDLAAGNATLDAARAQATAISPA...


In [6]:
# ============================================================
# Cell 4: Load ESMFold model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device != "cuda":
    print("WARNING: No GPU detected. This may be very slow or fail on free Colab CPU.")

gc.collect()

if device == "cuda":
    torch.cuda.empty_cache()

model = torch.load(
    MODEL_NAME,
    map_location=device,
    weights_only=False
)

model = model.eval()

if device == "cuda":
    model = model.cuda()
    torch.cuda.empty_cache()

gc.collect()

print("✓ ESMFold model loaded")

Device: cuda
✓ ESMFold model loaded


In [7]:
# ============================================================
# Cell 5: Predict binder structures with ESMFold
# ============================================================

if device != "cuda":
    raise RuntimeError(
        "This ESMFold notebook requires a GPU. "
        "CPU inference is not supported because the model uses CUDA-only attention kernels."
    )

prediction_rows = []

summary_csv = f"{OUTPUT_DIR}/esmfold_binder_predictions.csv"
partial_csv = f"{OUTPUT_DIR}/esmfold_binder_predictions_partial.csv"

if os.path.exists(summary_csv):
    existing = pd.read_csv(summary_csv)
    completed_ids = set(existing["seq_id"])
    prediction_rows = existing.to_dict("records")
    print(f"Loaded {len(completed_ids)} existing completed predictions")
else:
    completed_ids = set()

for i, row in df_candidates.iterrows():

    backbone = row["backbone"]
    seq = row["binder_sequence"]
    seq_id = f"{backbone}_rank{i}"

    output_pdb = f"{OUTPUT_DIR}/{seq_id}.pdb"

    if seq_id in completed_ids or os.path.exists(output_pdb):
        print(f"Skipping existing: {seq_id}")
        continue

    try:
        with torch.no_grad():
            pdb_string = model.infer_pdb(seq)

        with open(output_pdb, "w") as f:
            f.write(pdb_string)

        plddt_values = []

        for line in pdb_string.splitlines():
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                plddt_values.append(float(line[60:66]))

        if len(plddt_values) == 0:
            raise ValueError("No CA pLDDT values found in ESMFold output")

        mean_plddt = sum(plddt_values) / len(plddt_values)

        prediction_rows.append({
            "seq_id": seq_id,
            "backbone": backbone,
            "binder_sequence": seq,
            "binder_length": len(seq),
            "mpnn_score": row["mpnn_score"],
            "global_score": row["global_score"],
            "mean_plddt": round(mean_plddt, 2),
            "predicted_pdb": output_pdb,
        })

        df_partial = pd.DataFrame(prediction_rows)
        df_partial.to_csv(partial_csv, index=False)
        df_partial.to_csv(summary_csv, index=False)

        print(f"{i+1}/{len(df_candidates)} {seq_id}: pLDDT={mean_plddt:.1f}")

        del pdb_string

        if device == "cuda":
            torch.cuda.empty_cache()

        gc.collect()

    except Exception as e:
        print(f"Failed {seq_id}: {e}")

df_predictions = pd.DataFrame(prediction_rows)
df_predictions.to_csv(summary_csv, index=False)
print(df_candidates[["backbone","mpnn_score"]])
print("✓ ESMFold binder predictions complete")
print(f"Saved: {summary_csv}")

display(df_predictions)

1/10 design_9_0_rank0: pLDDT=67.6
2/10 design_19_0_rank1: pLDDT=67.7
3/10 design_0_0_rank2: pLDDT=80.0
4/10 design_7_0_rank3: pLDDT=96.7
5/10 design_6_0_rank4: pLDDT=93.4
6/10 design_4_0_rank5: pLDDT=82.0
7/10 design_18_0_rank6: pLDDT=95.2
8/10 design_22_0_rank7: pLDDT=76.7
9/10 design_23_0_rank8: pLDDT=95.6
10/10 design_1_0_rank9: pLDDT=95.1
      backbone  mpnn_score
0   design_9_0      0.8210
1  design_19_0      0.8493
2   design_0_0      0.8645
3   design_7_0      0.8931
4   design_6_0      0.9113
5   design_4_0      0.9173
6  design_18_0      0.9340
7  design_22_0      0.9394
8  design_23_0      0.9519
9   design_1_0      0.9583
✓ ESMFold binder predictions complete
Saved: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/esmfold_predictions/esmfold_binder_predictions.csv


,seq_id,backbone,binder_sequence,binder_length,mpnn_score,global_score,mean_plddt,predicted_pdb
0,design_9_0_rank0,design_9_0,AAVAAAVAQAHLALARAVADASPEEQAAAAARANAAAVAAAADPAA...,70,0.8210,1.3588,67.59,/content/drive/Othercomputers/My Mac/pdl1-mini...
1,design_19_0_rank1,design_19_0,SEAARAAAAAAAAAGAAAAAATACPAAAAAAAAAAAGAAAASAEGG...,70,0.8493,1.3756,67.66,/content/drive/Othercomputers/My Mac/pdl1-mini...
2,design_0_0_rank2,design_0_0,MEASDKLAAALAALAAAAAAAAAAGDLAAGRAALAAAGDAAAAELA...,70,0.8645,1.3683,79.98,/content/drive/Othercomputers/My Mac/pdl1-mini...
3,design_7_0_rank3,design_7_0,AAAARAALLAAAAAAAAAAAAATAALAAAAAELTAAGNAAGAAALK...,70,0.8931,1.3808,96.69,/content/drive/Othercomputers/My Mac/pdl1-mini...
4,design_6_0_rank4,design_6_0,ATCAALAERAEAAAKAYGLALVAGNATAAAEAAAAAAAVLAEAAAD...,70,0.9113,1.3849,93.39,/content/drive/Othercomputers/My Mac/pdl1-mini...
5,design_4_0_rank5,design_4_0,SEEKRRAAEEARARARERIAALDPANAERIMAAIEEVSPGGARAIG...,70,0.9173,1.3692,82.02,/content/drive/Othercomputers/My Mac/pdl1-mini...
6,design_18_0_rank6,design_18_0,AAAAAAAAAAAAALASAVAGAAAAGELNPKAKPAVDAAIAALAAAF...,70,0.9340,1.4174,95.15,/content/drive/Othercomputers/My Mac/pdl1-mini...
7,design_22_0_rank7,design_22_0,LDEQDARACLRAAENALAAAKANPATALRDLVAAAWFLARAAEALD...,70,0.9394,1.4228,76.68,/content/drive/Othercomputers/My Mac/pdl1-mini...
8,design_23_0_rank8,design_23_0,MGEGVRKVGEGCELLRRAVALAPEDPAAAAALAARGDELIAAGMAT...,70,0.9519,1.4143,95.62,/content/drive/Othercomputers/My Mac/pdl1-mini...
9,design_1_0_rank9,design_1_0,SEAYKQALATLAAGIAAATKAAATDLAAGNATLDAARAQATAISPA...,70,0.9583,1.4177,95.09,/content/drive/Othercomputers/My Mac/pdl1-mini...


In [8]:
# ============================================================
# Cell 6: Quick validation summary
# ============================================================

if "df_predictions" not in globals():
    df_predictions = pd.read_csv(f"{OUTPUT_DIR}/esmfold_binder_predictions.csv")

print(df_predictions["mean_plddt"].describe())

df_pass = df_predictions[df_predictions["mean_plddt"] >= 80].copy()

print(f"Candidates with mean pLDDT >= 80: {len(df_pass)}/{len(df_predictions)}")

display(
    df_predictions
    .sort_values("mean_plddt", ascending=False)
    [
        [
            "seq_id",
            "backbone",
            "binder_length",
            "mpnn_score",
            "global_score",
            "mean_plddt",
            "predicted_pdb"
        ]
    ]
)

count    10.000000
mean     84.987000
std      11.698758
min      67.590000
25%      77.505000
50%      87.705000
75%      95.135000
max      96.690000
Name: mean_plddt, dtype: float64
Candidates with mean pLDDT >= 80: 6/10


,seq_id,backbone,binder_length,mpnn_score,global_score,mean_plddt,predicted_pdb
3,design_7_0_rank3,design_7_0,70,0.8931,1.3808,96.69,/content/drive/Othercomputers/My Mac/pdl1-mini...
8,design_23_0_rank8,design_23_0,70,0.9519,1.4143,95.62,/content/drive/Othercomputers/My Mac/pdl1-mini...
6,design_18_0_rank6,design_18_0,70,0.9340,1.4174,95.15,/content/drive/Othercomputers/My Mac/pdl1-mini...
9,design_1_0_rank9,design_1_0,70,0.9583,1.4177,95.09,/content/drive/Othercomputers/My Mac/pdl1-mini...
4,design_6_0_rank4,design_6_0,70,0.9113,1.3849,93.39,/content/drive/Othercomputers/My Mac/pdl1-mini...
5,design_4_0_rank5,design_4_0,70,0.9173,1.3692,82.02,/content/drive/Othercomputers/My Mac/pdl1-mini...
2,design_0_0_rank2,design_0_0,70,0.8645,1.3683,79.98,/content/drive/Othercomputers/My Mac/pdl1-mini...
7,design_22_0_rank7,design_22_0,70,0.9394,1.4228,76.68,/content/drive/Othercomputers/My Mac/pdl1-mini...
1,design_19_0_rank1,design_19_0,70,0.8493,1.3756,67.66,/content/drive/Othercomputers/My Mac/pdl1-mini...
0,design_9_0_rank0,design_9_0,70,0.8210,1.3588,67.59,/content/drive/Othercomputers/My Mac/pdl1-mini...


In [9]:
# ============================================================
# Cell 7: Save high-confidence candidates
# ============================================================

PASS_PLDDT = 80

df_high_conf = df_predictions[
    df_predictions["mean_plddt"] >= PASS_PLDDT
].copy()

high_conf_csv = f"{OUTPUT_DIR}/high_confidence_esmfold_candidates.csv"

df_high_conf.to_csv(high_conf_csv, index=False)

print(f"Saved {len(df_high_conf)} high-confidence candidates")
print(high_conf_csv)

display(df_high_conf)

Saved 6 high-confidence candidates
/content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/esmfold_predictions/high_confidence_esmfold_candidates.csv


,seq_id,backbone,binder_sequence,binder_length,mpnn_score,global_score,mean_plddt,predicted_pdb
3,design_7_0_rank3,design_7_0,AAAARAALLAAAAAAAAAAAAATAALAAAAAELTAAGNAAGAAALK...,70,0.8931,1.3808,96.69,/content/drive/Othercomputers/My Mac/pdl1-mini...
4,design_6_0_rank4,design_6_0,ATCAALAERAEAAAKAYGLALVAGNATAAAEAAAAAAAVLAEAAAD...,70,0.9113,1.3849,93.39,/content/drive/Othercomputers/My Mac/pdl1-mini...
5,design_4_0_rank5,design_4_0,SEEKRRAAEEARARARERIAALDPANAERIMAAIEEVSPGGARAIG...,70,0.9173,1.3692,82.02,/content/drive/Othercomputers/My Mac/pdl1-mini...
6,design_18_0_rank6,design_18_0,AAAAAAAAAAAAALASAVAGAAAAGELNPKAKPAVDAAIAALAAAF...,70,0.9340,1.4174,95.15,/content/drive/Othercomputers/My Mac/pdl1-mini...
8,design_23_0_rank8,design_23_0,MGEGVRKVGEGCELLRRAVALAPEDPAAAAALAARGDELIAAGMAT...,70,0.9519,1.4143,95.62,/content/drive/Othercomputers/My Mac/pdl1-mini...
9,design_1_0_rank9,design_1_0,SEAYKQALATLAAGIAAATKAAATDLAAGNATLDAARAQATAISPA...,70,0.9583,1.4177,95.09,/content/drive/Othercomputers/My Mac/pdl1-mini...


In [12]:
# ============================================================
# Cell 5: Calculate RMSD between ESMFold prediction and RFdiffusion backbone
# ============================================================

import os
import glob
import pandas as pd
from Bio.PDB import PDBParser, Superimposer

parser = PDBParser(QUIET=True)

# RFdiffusion outputs
BACKBONE_DIR = "/content/drive/MyDrive/Colab Notebooks/rfdiffusion_outputs"

# ESMFold predictions table
PREDICTION_CSV = f"{OUTPUT_DIR}/esmfold_binder_predictions.csv"

if not os.path.exists(PREDICTION_CSV):
    PREDICTION_CSV = f"{OUTPUT_DIR}/esmfold_binder_predictions_partial.csv"

df_predictions = pd.read_csv(PREDICTION_CSV)

rmsd_results = []

for _, row in df_predictions.iterrows():

    seq_id = row["seq_id"]
    backbone = row["backbone"]

    predicted_pdb = row["predicted_pdb"]
    designed_pdb = f"{BACKBONE_DIR}/{backbone}.pdb"

    if not os.path.exists(predicted_pdb):
        print(f"Missing predicted PDB: {predicted_pdb}")
        continue

    if not os.path.exists(designed_pdb):
        print(f"Missing designed PDB: {designed_pdb}")
        continue

    try:
        pred_struct = parser.get_structure("pred", predicted_pdb)
        design_struct = parser.get_structure("design", designed_pdb)

        # ESMFold binder-only prediction usually has one chain
        pred_chain = list(pred_struct[0].get_chains())[0]

        # RFdiffusion output: binder = chain B, target = chain A
        design_chain = design_struct[0]["B"]

        pred_ca = [
            atom
            for residue in pred_chain.get_residues()
            for atom in residue.get_atoms()
            if atom.name == "CA"
        ]

        design_ca = [
            atom
            for residue in design_chain.get_residues()
            for atom in residue.get_atoms()
            if atom.name == "CA"
        ]

        n_atoms = min(len(pred_ca), len(design_ca))

        if n_atoms < 10:
            print(f"Too few atoms for RMSD: {seq_id}")
            continue

        pred_ca = pred_ca[:n_atoms]
        design_ca = design_ca[:n_atoms]

        sup = Superimposer()
        sup.set_atoms(design_ca, pred_ca)

        rmsd_results.append({
            "seq_id": seq_id,
            "backbone": backbone,
            "mean_plddt": row["mean_plddt"],
            "mpnn_score": row["mpnn_score"],
            "global_score": row["global_score"],
            "rmsd": round(sup.rms, 2),
            "n_aligned": n_atoms,
            "predicted_pdb": predicted_pdb,
            "designed_pdb": designed_pdb,
        })

    except Exception as e:
        print(f"RMSD failed for {seq_id}: {e}")

df_rmsd = pd.DataFrame(rmsd_results)

rmsd_csv = f"{OUTPUT_DIR}/validation_results.csv"
df_rmsd.to_csv(rmsd_csv, index=False)

print(f"Saved RMSD validation results to: {rmsd_csv}")
print(f"Validated structures: {len(df_rmsd)}")

if len(df_rmsd) > 0:
    display(
        df_rmsd
        .sort_values(["rmsd", "mean_plddt"], ascending=[True, False])
    )

Saved RMSD validation results to: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/esmfold_predictions/validation_results.csv
Validated structures: 10


,seq_id,backbone,mean_plddt,mpnn_score,global_score,rmsd,n_aligned,predicted_pdb,designed_pdb
9,design_1_0_rank9,design_1_0,95.09,0.9583,1.4177,0.44,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
3,design_7_0_rank3,design_7_0,96.69,0.8931,1.3808,0.88,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
8,design_23_0_rank8,design_23_0,95.62,0.9519,1.4143,0.90,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
4,design_6_0_rank4,design_6_0,93.39,0.9113,1.3849,0.97,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
7,design_22_0_rank7,design_22_0,76.68,0.9394,1.4228,1.04,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
0,design_9_0_rank0,design_9_0,67.59,0.8210,1.3588,2.59,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
1,design_19_0_rank1,design_19_0,67.66,0.8493,1.3756,2.89,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
2,design_0_0_rank2,design_0_0,79.98,0.8645,1.3683,2.93,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
6,design_18_0_rank6,design_18_0,95.15,0.9340,1.4174,3.35,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
5,design_4_0_rank5,design_4_0,82.02,0.9173,1.3692,11.89,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...


In [13]:
# ============================================================
# Cell 6: Filter validated designs
# ============================================================

PLDDT_THRESHOLD = 80
RMSD_THRESHOLD = 2.0

df_validated = df_rmsd[
    (df_rmsd["mean_plddt"] >= PLDDT_THRESHOLD) &
    (df_rmsd["rmsd"] <= RMSD_THRESHOLD)
].copy()

validated_csv = f"{OUTPUT_DIR}/validated_designs.csv"
df_validated.to_csv(validated_csv, index=False)

print(f"Validated designs: {len(df_validated)}/{len(df_rmsd)}")
print(f"Saved to: {validated_csv}")

if len(df_validated) > 0:
    display(
        df_validated
        .sort_values(["rmsd", "mean_plddt"], ascending=[True, False])
    )
else:
    print("No designs passed both pLDDT and RMSD thresholds.")

Validated designs: 4/10
Saved to: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/esmfold_predictions/validated_designs.csv


,seq_id,backbone,mean_plddt,mpnn_score,global_score,rmsd,n_aligned,predicted_pdb,designed_pdb
9,design_1_0_rank9,design_1_0,95.09,0.9583,1.4177,0.44,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
3,design_7_0_rank3,design_7_0,96.69,0.8931,1.3808,0.88,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
8,design_23_0_rank8,design_23_0,95.62,0.9519,1.4143,0.90,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...
4,design_6_0_rank4,design_6_0,93.39,0.9113,1.3849,0.97,70,/content/drive/Othercomputers/My Mac/pdl1-mini...,/content/drive/MyDrive/Colab Notebooks/rfdiffu...


In [14]:
# ============================================================
# Cell 7: Optional complex prediction placeholder
# ============================================================

print(
    "Complex prediction is not run in this notebook yet. "
    "Use ColabFold/AlphaFold-Multimer or an ESMFold complex run later."
)

Complex prediction is not run in this notebook yet. Use ColabFold/AlphaFold-Multimer or an ESMFold complex run later.
